# DefectosCafeVerde grouped - DCWCF1 seed 42

Fresh transfer of classification-only chromatic-wavelet conditioning. Native RGB localization is unchanged; the grouped test split is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
ARM='DCWCF1'; BRANCH='codex/defectoscafeverde-cwcf-direct'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab sebelum Run All')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_ARCHIVE_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
drive_roots=[Path('/content/drive/MyDrive'),*Path('/content/drive/.shortcut-targets-by-id').glob('*')]
matches=sorted({path for root in drive_roots for path in root.rglob(ARCHIVE_NAME)})
parts=sorted({path for root in drive_roots for path in root.rglob(ARCHIVE_NAME+'.chunk-*')})
if len(matches)==1:
    ARCHIVE=matches[0]
elif parts:
    expected=[f'{ARCHIVE_NAME}.chunk-{index:03d}' for index in range(11)]
    if [part.name for part in parts]!=expected: raise RuntimeError(f'Potongan archive tidak lengkap/berurutan: {[part.name for part in parts]}')
    ARCHIVE=WORK/ARCHIVE_NAME
    with ARCHIVE.open('wb') as target:
        for part in parts:
            with part.open('rb') as source: shutil.copyfileobj(source,target,length=8*1024*1024)
else: raise FileNotFoundError(f'Tidak menemukan {ARCHIVE_NAME} atau 11 chunk di Drive')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('SHA archive DefectosCafeVerde tidak cocok')
print('ARCHIVE VALID:',ARCHIVE)

In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'; allowed_files={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        if not (first in {'train','val'} or relative in allowed_files): continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True); source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE - STOP')
import yaml
payload=yaml.safe_load((DATA/'data.yaml').read_text()); payload.pop('test',None)
payload['path']=str(DATA); payload['train']='train/images'; payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))),'TEST:',(DATA/'test').exists())

In [ ]:
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=(REPO/'yolo26n.pt').resolve()
OUT=Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments/defectoscafeverde-cwcf-direct-v1'); OUT.mkdir(parents=True,exist_ok=True)
from coffee_detector.experiments.run_defectoscafeverde_cwcf_direct import run_static_preflight, validate_development_dataset
contract=validate_development_dataset(DATA,AUDIT)
static=run_static_preflight(PRETRAINED,OUT/f'static_preflight_{ARM}_notebook.json',seed=42)
print('PARAMETERS:',{'native':static['native_parameters'],'candidate':static['candidate_parameters'],'added':static['candidate_added_parameters']})
print('DATA GATES:',contract['gates']); print('STATIC GATES:',static['gates']); print('DECISION:',static['decision'])
assert static['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'

In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_cwcf_direct','--arm',ARM,'--data-root',str(DATA),'--grouped-audit',str(AUDIT),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')

In [ ]:
from coffee_detector.experiments.run_defectoscafeverde_cwcf_direct import build_decision
NATIVE=OUT/'val_reports/D0DIRECT_seed42_result.json'; CANDIDATE=OUT/'val_reports/DCWCF1_seed42_result.json'
if NATIVE.is_file() and CANDIDATE.is_file():
    decision=build_decision(NATIVE,CANDIDATE,OUT/'defectoscafeverde_dcwcf_seed42_decision.json')
    print('VALUES:',decision['values']); print('DELTAS:',decision['deltas']); print('SCREEN:',decision['screen']); print('TEST:',decision['test_opened'])
else: print('Arm pasangan belum selesai. Jalankan ulang cell ini setelah D0DIRECT dan DCWCF1 tersedia.')